<a href="https://colab.research.google.com/github/ckrickyh/pythonTools/blob/main/TMCP_F2_DataSummarization_Ver4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import geopandas as gpd
import pandas as pd
import json
import xlwings as xw
import zipfile
import pathlib
import numpy as np
from pathlib import Path

In [ ]:
folderpath = input('e.g. L:\LU\TRAM 10 Form 2\Batch 2\Done_zip,pdf,attachment')

e.g. L:\LU\TRAM 10 Form 2\Batch 2\Done_zip,pdf,attachment L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (202410-202412)_F2\202410\20241028_W SWT


# Interal docPath Setting

In [ ]:
CSV2Masterjsondata = r'C:\TMCP\CSVToMasterValue.json'
tmcpCSVpath = r'C:\TMCP\CSVUnZIPTemp'

#Export Result location
xlExportPath = r'C:\Users\ckho\Desktop\tmcp\tram10f2summary.xlsx' #小心, 這FILE 以f2 TARGET NO. 做BASE (不是 F2 TREE ID 做 BASE), 所以DATA 會 多於一個 TREE ID
xlErrPath = r'C:\Users\ckho\Desktop\tmcp\tram10f2errzip.xlsx'

# Null setting

In [ ]:
df_f2basicAll = []
count = 0
num = 0
d = None
d = {}
dall = None
dall = {}
fllist =[]

# Read CSVtoMasterValue json

In [ ]:
#CSV2Masterjsondata = r'C:\TMCP\CSVToMasterValue.json'

pathlist = str(Path(CSV2Masterjsondata).parent)
pathlist

with open(CSV2Masterjsondata) as f:
    json_data = f.read()
    data = json.loads(json_data)
    #print(data)
    df_csv = pd.json_normalize(data)

df_csv['link'] = pathlist + '\\' +df_csv['FileName']
df_csv.head()

,FileName,MasterType,IsSystem,link
0,TBL_ADVANCED_ASSESSMENT.csv,AdvancedAssessment,false,C:\TMCP\TBL_ADVANCED_ASSESSMENT.csv
1,TBL_ASPECT.csv,Aspect,false,C:\TMCP\TBL_ASPECT.csv
2,TBL_AUDITOR.csv,,false,C:\TMCP\TBL_AUDITOR.csv
3,TBL_CONTRACT.csv,,False,C:\TMCP\TBL_CONTRACT.csv
4,TBL_CROWN_DENSITY_PERCENT.csv,CrownDensityD,false,C:\TMCP\TBL_CROWN_DENSITY_PERCENT.csv


# Read CSV decoding

In [ ]:
#tmcpCSVpath = r'C:\TMCP\CSVUnZIPTemp'

#create dataframe blank in dic
dCsv=None
dCsv={}

path = pathlib.Path(tmcpCSVpath)
files = path.glob("**/TBL*.csv")

#codetablelist
for i in files:
    dCsv[i.stem + i.suffix] = pd.read_csv(i, sep='\t')
dCsv.keys()

dict_keys(['TBL_ADVANCED_ASSESSMENT.csv', 'TBL_ASPECT.csv', 'TBL_AUDITOR.csv', 'TBL_CONTRACT.csv', 'TBL_CROWN_DEFOLIATION_PERCENT.csv', 'TBL_CROWN_DENSITY_PERCENT.csv', 'TBL_CROWN_DIEBACK.csv', 'TBL_CROWN_FOLIAGE.csv', 'TBL_CROWN_LEAF_SIZE.csv', 'TBL_CROWN_LIVE_CROWN.csv', 'TBL_CROWN_LOAD.csv', 'TBL_CROWN_LOAD_DENSITY.csv', 'TBL_DEPARTMENT.csv', 'TBL_DISTRICT.csv', 'TBL_FAILURE_STATUS.csv', 'TBL_FAILURE_TYPE.csv', 'TBL_INSPECTION_FREQUENCEY.csv', 'TBL_INSPECTION_TIME.csv', 'TBL_INSPECTOR.csv', 'TBL_IRRIGATION.csv', 'TBL_LOCATION_TYPE.csv', 'TBL_MASTER_ZONE.csv', 'TBL_MITIGATION_MEASURE.csv', 'TBL_PEST_DISEASE.csv', 'TBL_PRUN_LION.csv', 'TBL_PRUN_REDUCTION.csv', 'TBL_REMEDIAL_ACTION.csv', 'TBL_RESIDUAL_RISK.csv', 'TBL_RESTRICTION_DRIPLINE.csv', 'TBL_RISK_CONSEQUENCE.csv', 'TBL_RISK_FAILURE.csv', 'TBL_RISK_FAILURE_IMPACT.csv', 'TBL_RISK_IMPACT.csv', 'TBL_RISK_RATING.csv', 'TBL_RISK_RATING_MATRIX.csv', 'TBL_ROOT_CONDITION.csv', 'TBL_ROOT_FAILURE_TYPE.csv', 'TBL_ROOT_FUNGAL.csv', 'TBL_SLOP

# Read zip

In [ ]:
fllist =[]
errlist =[]

#folderpath = r'C:\Users\ckho\Desktop\trialtmcp'
path = Path(folderpath)
files = path.glob("**/*.zip")

for zip in files:
    fllist.append(zip)

for zip in fllist:
    print('File. ' + str(num+1) + ' / ' + str(len(fllist)))
    print(zip)

    try:
        with zipfile.ZipFile(zip, "r") as z:
            print(z.namelist())

            for filename in z.namelist():
                print('z: ' + filename)


                #===========================================================================================database and f2
                #if 'json' in filename and 'Version' in filename and '/' not in filename:      #===========read version and mapping database
                if 'VersionAndMapping.json' in filename:
                    f2 = filename
                    jsondata = f2
                    print(f2)

                    pathlist = str(Path(jsondata).parent)

                    with z.open(jsondata) as f:
                        json_data = f.read()
                        datameta = json.loads(json_data)
                        print('Y')
                        #print(datameta)
                        #df_summary = pd.json_normalize(datameta)

                        df_dbmasterValue = pd.json_normalize(datameta, record_path = ['masterValueList'])  #=======gen version and mapping, Read mastervaluelist data, then lookup

                        #cb clickbox
                        df_decode1 = df_dbmasterValue[df_dbmasterValue.Code.str.startswith('cb')]
                        df_decode1 = df_decode1.rename(columns={"Code": "Result", "MasterTypeCode": "Code"})
                        #df_decode1.head()

                        #form2.SoilCrackOrCrackBehindLean
                        #form2.Status
                        otherlist = ['SCOrCBLean','Status']
                        df_decode2 = df_dbmasterValue[(df_dbmasterValue.MasterTypeCode.isin(otherlist))]
                        df_decode2 = df_decode2.rename(columns={"Code": "Result", "MasterTypeCode": "Code"})
                        #df_decode2.head()

                        #Merge df_dbmasterValue and df_csv  dataset
                        dflink = pd.merge(df_dbmasterValue,df_csv, left_on='MasterTypeCode', right_on='MasterType')
                        ##dflink['Code'].astype(str)
                        #dflink.head()

                        #Decoding Mapping and Version recorded in 'Result Column'
                        resultlist = []
                        for index, row in dflink.iterrows():
                            #print(row['Code'])
                            selectData = [int(row['Code'])]
                            #print(dCsv[row["FileName"]])
                            dfa = dCsv[row["FileName"]]
                            dfacode = dCsv[row["FileName"]]['CSV_CODE']
                            result = dfa[(dfacode.isin(selectData))].iloc[0,3]
                            resultlist.append(result)
                            #df2[(df2['ID'].isin(df1['ID']))]

                        dflink['Result'] = resultlist

                        dflink = pd.concat([dflink, df_decode1, df_decode2], ignore_index=True)   #Decoding
                        #dflink


                        #============== 全區tree dot 地圖, read treelist , use "ID" connect TreeID, DEPTTreeID, SpeciesCode, SpeciesTypeCode, MasterZoneID, SubZoneID, TriageColourCode, TriageColourTypeCode, X, Y, isNewTree
                        df_dbtreeList = pd.json_normalize(datameta, record_path = ['treeList'])

                #============================================================================================f2
                elif 'F2' in str(filename) and '.json' in str(filename): #===================read f2
                    f2 = filename
                    jsondata = f2
                    print('Y2')
                    print(jsondata)

                    with z.open(jsondata) as f:
                        json_data = f.read()
                        data = json.loads(json_data)
                        #print(data)
                        df_f2 = pd.json_normalize(data)

                        #==============================================================================F2 Basic infomation
                        f2basiclist =  df_f2.columns.to_list()[18:] #===========F2 Basic infomation

                        df_f2basic = df_f2[f2basiclist].replace(dflink.ID.tolist(),  dflink.Result.tolist())
                        #df_f2basic

                        TreeID = df_f2basic['form2.DEPTTreeID'].iloc[0] #========get TreeID
                        #X = df_f2basic['form2.CoordinateX'].iloc[0] #========get X
                        #Y = df_f2basic['form2.CoordinateY'].iloc[0] #========get Y
                        OverallComment = df_f2basic['form2.MitigationMeasureRemark'].iloc[0] #========get Comment
                        TMCPTreeID = pd.DataFrame(df_f2basic['form2.TreeGUID']).replace(df_dbtreeList.ID.tolist(),  df_dbtreeList.TreeID.tolist())['form2.TreeGUID'].iloc[0] #========get TMCPTreeID
                        print(TMCPTreeID)


                        #df_f2basicAll = []   #=========empty f2basicall
                        #count = 0
                        if count == 0:
                            df_f2basicAll = pd.DataFrame(columns = df_f2basic.columns.tolist())
                        df_f2basicAll = pd.concat([df_f2basicAll, df_f2basic])
                        #print(df_f2basicAll)




                        #==============================================================================f2 advance infomation with decoding , link with csv
                        #num = 0

                        #create dataframe blank in dic
                        d = None
                        d = {}

                        f2sectionlist = df_f2.columns.to_list()[0:17]
                        #f2sectionlist.extend(['form2.TreeGUID'])

                        #codetablelist
                        for i in f2sectionlist:
                            d[i] = pd.json_normalize(data, record_path = [i])
                            d[i]['TreeID']=TreeID
                            #d[i]['X']=X
                            #d[i]['Y']=Y
                            d[i]['TMCPTreeID']=TMCPTreeID
                            d[i]['OverallComment']=OverallComment
                            d[i]['fldPath'] = str(zip)

                            #print(num)

                            ###Decoding
                            d[i] = d[i].replace(dflink.ID.tolist(),  dflink.Result.tolist())
                            #d[i] = d[i].replace(df_dbtreeList.ID.tolist(),  df_dbtreeList.TreeID.tolist()) #df_dbtreelist decode form2.TreeGUID

                            #Append gp
                            if num == 0:
                                dall[i] = pd.DataFrame(columns = d[i].columns.tolist())
                            dall[i] = pd.concat([dall[i],d[i]])


                            #d.keys()
                            #dall.keys()

            #=============================run 完一個 file 後
            count = count + 1
            num = num +1
    except:
        errlist.append(str(zip))
        pass

File. 1 / 13
L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (202410-202412)_F2\202410\20241028_W SWT\ETR0610143\20250113163616_ETR0610143_11SW-A-R577_0_20241028.zip
['VersionAndMapping.json', 'F2_ETR0610143_11SW-A-R577_0_20241028/AUDIT/', 'F2_ETR0610143_11SW-A-R577_0_20241028/ETR0610143_11SW-A-R577_0_20241028.json', 'F2_ETR0610143_11SW-A-R577_0_20241028/VersionAndMapping.json', 'F2_ETR0610143_11SW-A-R577_0_20241028/MAP/map.png', 'F2_ETR0610143_11SW-A-R577_0_20241028/OTHER/20240429_ETR0610143_Photo.pdf', 'F2_ETR0610143_11SW-A-R577_0_20241028/OTHER/20241028_ETR0610143_Photo.pdf', 'F2_ETR0610143_11SW-A-R577_0_20241028/OTHER/after pruning in Nov 2024.png', 'F2_ETR0610143_11SW-A-R577_0_20241028/PHOTO/Profile.jpg', 'F2_ETR0610143_11SW-A-R577_0_20241028/PHOTO/TimePhoto_20241028_150957.jpg']
z: VersionAndMapping.json
VersionAndMapping.json
Y
z: F2_ETR0610143_11SW-A-R577_0_20241028/AUDIT/
z: F2_ETR0610143_11SW-A-R577_0_20241028/ETR0610143_11SW-A-R577_0_20241028.json
Y2
F2_ETR0610143_11SW-A-R577_0_

C:\Users\ckho\AppData\Local\Temp\ipykernel_2536\2792450771.py:110: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  df_f2basicAll = pd.concat([df_f2basicAll, df_f2basic])
C:\Users\ckho\AppData\Local\Temp\ipykernel_2536\2792450771.py:145: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  dall[i] = pd.concat([dall[i],d[i]])


z: F2_ETR0610144_11SW-A-R577_0_20241028/VersionAndMapping.json
F2_ETR0610144_11SW-A-R577_0_20241028/VersionAndMapping.json
Y
z: F2_ETR0610144_11SW-A-R577_0_20241028/MAP/map.png
z: F2_ETR0610144_11SW-A-R577_0_20241028/OTHER/20241204_ETR0610144_Photo.pdf
z: F2_ETR0610144_11SW-A-R577_0_20241028/PHOTO/Profile.jpg
z: F2_ETR0610144_11SW-A-R577_0_20241028/PHOTO/TimePhoto_20241028_153815.jpg
z: F2_ETR0610144_11SW-A-R577_0_20241028/PHOTO/TimePhoto_20241028_153820.jpg
File. 3 / 13
L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (202410-202412)_F2\202410\20241028_W SWT\ETR0610145\20250113165856_ETR0610145_11SW-A-R577_0_20241028.zip
['VersionAndMapping.json', 'F2_ETR0610145_11SW-A-R577_0_20241028/AUDIT/', 'F2_ETR0610145_11SW-A-R577_0_20241028/ETR0610145_11SW-A-R577_0_20241028.json', 'F2_ETR0610145_11SW-A-R577_0_20241028/VersionAndMapping.json', 'F2_ETR0610145_11SW-A-R577_0_20241028/MAP/map.png', 'F2_ETR0610145_11SW-A-R577_0_20241028/OTHER/20241028_ETR0610145_Photo.pdf', 'F2_ETR0610145_11SW-A-R577_0_20

C:\Users\ckho\AppData\Local\Temp\ipykernel_2536\2792450771.py:110: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  df_f2basicAll = pd.concat([df_f2basicAll, df_f2basic])
C:\Users\ckho\AppData\Local\Temp\ipykernel_2536\2792450771.py:145: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  dall[i] = pd.concat([dall[i],d[i]])


z: F2_ETR0610145_11SW-A-R577_0_20241028/VersionAndMapping.json
F2_ETR0610145_11SW-A-R577_0_20241028/VersionAndMapping.json
Y
z: F2_ETR0610145_11SW-A-R577_0_20241028/MAP/map.png
z: F2_ETR0610145_11SW-A-R577_0_20241028/OTHER/20241028_ETR0610145_Photo.pdf
z: F2_ETR0610145_11SW-A-R577_0_20241028/OTHER/after.jpg
z: F2_ETR0610145_11SW-A-R577_0_20241028/PHOTO/Profile.jpg
z: F2_ETR0610145_11SW-A-R577_0_20241028/PHOTO/Profile.png
z: F2_ETR0610145_11SW-A-R577_0_20241028/PHOTO/TimePhoto_20241028_151733.jpg
File. 4 / 13
L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (202410-202412)_F2\202410\20241028_W SWT\ETR0610147\20250113171542_ETR0610147_11SW-A-R577_0_20241028.zip
['VersionAndMapping.json', 'F2_ETR0610147_11SW-A-R577_0_20241028/AUDIT/', 'F2_ETR0610147_11SW-A-R577_0_20241028/ETR0610147_11SW-A-R577_0_20241028.json', 'F2_ETR0610147_11SW-A-R577_0_20241028/VersionAndMapping.json', 'F2_ETR0610147_11SW-A-R577_0_20241028/MAP/map.png', 'F2_ETR0610147_11SW-A-R577_0_20241028/OTHER/20241028_ETR0610147_Photo

...

z: F2_hyd_hk_11sw_a_r614_0_wt1_11SW-A-R614_0_20241028/VersionAndMapping.json
F2_hyd_hk_11sw_a_r614_0_wt1_11SW-A-R614_0_20241028/VersionAndMapping.json
Y
z: F2_hyd_hk_11sw_a_r614_0_wt1_11SW-A-R614_0_20241028/MAP/map.png
z: F2_hyd_hk_11sw_a_r614_0_wt1_11SW-A-R614_0_20241028/OTHER/20241028_RTR0033984_Photo.pdf
z: F2_hyd_hk_11sw_a_r614_0_wt1_11SW-A-R614_0_20241028/PHOTO/Profile.jpg
z: F2_hyd_hk_11sw_a_r614_0_wt1_11SW-A-R614_0_20241028/PHOTO/TimePhoto_20241028_141345.jpg
File. 12 / 13
L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (202410-202412)_F2\202410\20241028_W SWT\RTR0033994\20250114112721_RTR0033994_11SW-A-R602_0_20241028.zip
['VersionAndMapping.json', 'F2_RTR0033994_11SW-A-R602_0_20241028/AUDIT/', 'F2_RTR0033994_11SW-A-R602_0_20241028/RTR0033994_11SW-A-R602_0_20241028.json', 'F2_RTR0033994_11SW-A-R602_0_20241028/VersionAndMapping.json', 'F2_RTR0033994_11SW-A-R602_0_20241028/MAP/map.jpg', 'F2_RTR0033994_11SW-A-R602_0_20241028/OTHER/20241028_RTR0033994_Photo.pdf', 'F2_RTR0033994_11SW-A-

C:\Users\ckho\AppData\Local\Temp\ipykernel_2536\2792450771.py:110: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  df_f2basicAll = pd.concat([df_f2basicAll, df_f2basic])
C:\Users\ckho\AppData\Local\Temp\ipykernel_2536\2792450771.py:145: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  dall[i] = pd.concat([dall[i],d[i]])


z: F2_RTR0033994_11SW-A-R602_0_20241028/VersionAndMapping.json
F2_RTR0033994_11SW-A-R602_0_20241028/VersionAndMapping.json
Y
z: F2_RTR0033994_11SW-A-R602_0_20241028/MAP/map.jpg
z: F2_RTR0033994_11SW-A-R602_0_20241028/OTHER/20241028_RTR0033994_Photo.pdf
z: F2_RTR0033994_11SW-A-R602_0_20241028/PHOTO/Profile.jpg
z: F2_RTR0033994_11SW-A-R602_0_20241028/PHOTO/TimePhoto_20241028_145743.jpg
File. 13 / 13
L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (202410-202412)_F2\202410\20241028_W SWT\RTR0034003\RTR0034003_11SW-A-R602_0_20241028\20250114114757_RTR0034003_11SW-A-R602_0_20241028.zip
['VersionAndMapping.json', 'F2_RTR0034003_11SW-A-R602_0_20241028/AUDIT/', 'F2_RTR0034003_11SW-A-R602_0_20241028/RTR0034003_11SW-A-R602_0_20241028.json', 'F2_RTR0034003_11SW-A-R602_0_20241028/VersionAndMapping.json', 'F2_RTR0034003_11SW-A-R602_0_20241028/MAP/map.jpg', 'F2_RTR0034003_11SW-A-R602_0_20241028/OTHER/20241028_RTR0034003_Photo.pdf', 'F2_RTR0034003_11SW-A-R602_0_20241028/PHOTO/Profile.jpg', 'F2_RTR0034003

C:\Users\ckho\AppData\Local\Temp\ipykernel_2536\2792450771.py:110: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  df_f2basicAll = pd.concat([df_f2basicAll, df_f2basic])
C:\Users\ckho\AppData\Local\Temp\ipykernel_2536\2792450771.py:145: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  dall[i] = pd.concat([dall[i],d[i]])


z: F2_RTR0034003_11SW-A-R602_0_20241028/VersionAndMapping.json
F2_RTR0034003_11SW-A-R602_0_20241028/VersionAndMapping.json
Y
z: F2_RTR0034003_11SW-A-R602_0_20241028/MAP/map.jpg
z: F2_RTR0034003_11SW-A-R602_0_20241028/OTHER/20241028_RTR0034003_Photo.pdf
z: F2_RTR0034003_11SW-A-R602_0_20241028/PHOTO/Profile.jpg
z: F2_RTR0034003_11SW-A-R602_0_20241028/PHOTO/TimePhoto_20240429_122335.jpg
z: F2_RTR0034003_11SW-A-R602_0_20241028/PHOTO/TimePhoto_20241028_142855.jpg


In [ ]:
#

triallist = ['VersionAndMapping.json', 'F2_RTR0004906_11SE-C-C39_0_20241126/AUDIT/', 'F2_RTR0004906_11SE-C-C39_0_20241126/OTHER/', 'F2_RTR0004906_11SE-C-C39_0_20241126/RTR0004906_11SE-C-C39_0_20241126.json', 'F2_RTR0004906_11SE-C-C39_0_20241126/VersionAndMapping.json', 'F2_RTR0004906_11SE-C-C39_0_20241126/MAP/RTR0004906 Tree Location.JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Concrete crack behind lean.JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Crown condition.JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Dead branch (1).JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Dead branch (2).JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Fungal fruiting bodies.JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Leaning.JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Overview.JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Root condition (1).JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Root condition (2).JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Root condition (3).JPG', 'F2_RTR0004906_11SE-C-C39_0_20241126/PHOTO/RTR0004906 - Trunk condition.JPG']

print('.json' in 'F2_RTR0004906_11SE-C-C39_0_20241126/RTR0004906_11SE-C-C39_0_20241126.json' and 'Version' not in 'F2_RTR0004906_11SE-C-C39_0_20241126/RTR0004906_11SE-C-C39_0_20241126.json')


True


In [ ]:
df_f2basic['form2.DEPTTreeID'].iloc[0]

'RTR0034003'

In [ ]:
data

{'form2': {'FormSubversion': '1.2',
  'ID': '94de1887-d843-4184-9317-0e0b7e24c624',
  'TreeGUID': 'fe40b278-d72a-44e3-8e51-abe363e3b56e',
  'ContractGUID': '36fe203a-543c-4d02-a68b-f592c337ce86',
  'FileRef': 'RTR0034003_11SW-AR602_20241028',
  'DepartmentGUID': 'b9604f7e-c99e-415b-8259-ffa2655783e4',
  'InspectionOfficerGUID': '2902076d-de6a-48ee-991d-e8aacdaf4e02',
  'DateOfInspection': '2024-10-28T14:11:27',
  'LastInspectionTime': '2024-04-29T00:00:00',
  'InspectionFrequency': '5ddc84b4-425b-40d3-995f-ecd8ae39bed3',
  'InspectionTimeSpent': '5e01fe1b-7915-494e-b1c3-9a8cd5876747',
  'LastExportDatetime': '2025-01-14T11:47:12.907',
  'LastImportDatetime': '2023-11-15T09:38:48.677',
  'LastUploadDatetime': None,
  'TreeSpecies': '5e10ffe7-7599-42e3-99f8-0479c0cda222',
  'TriageCategory': '2e25e614-714e-411b-9770-2fde7d96f153',
  'TreeHeight': 13.0,
  'CrownSpread': 9.0,
  'NoOfTrunk': 1,
  'DBHOfTreeTrunk1': 610.0,
  'DBHOfTreeTrunk2': None,
  'DBHOfTreeTrunk3': None,
  'DBHOfTreeTru

In [ ]:
df_f2.columns.to_list()

['attachment',
 'tblLocationTypeList',
 'tblForm2TargetAssessmentList',
 'tblForm2BranchConditionsList',
 'tblForm2MitigationMeasuresList',
 'tblForm2PruningHistoryList',
 'tblForm2RootConditionsList',
 'tblForm2TrunkConditionList',
 'tblForm2TreeOnTreeRegisterList',
 'tblForm2InspectionLimitationsList',
 'tblForm2DiebackTwigsTypeList',
 'tblForm2LeanList',
 'tblForm2SiteChangesList',
 'tblForm2SoilConditionsList',
 'tblForm2TopographyList',
 'tblForm2AuditCommentList',
 'auditTabs',
 'TreeID',
 'MasterZoneNo',
 'DateOfInspection',
 'form2.FormSubversion',
 'form2.ID',
 'form2.TreeGUID',
 'form2.ContractGUID',
 'form2.FileRef',
 'form2.DepartmentGUID',
 'form2.InspectionOfficerGUID',
 'form2.DateOfInspection',
 'form2.LastInspectionTime',
 'form2.InspectionFrequency',
 'form2.InspectionTimeSpent',
 'form2.LastExportDatetime',
 'form2.LastImportDatetime',
 'form2.LastUploadDatetime',
 'form2.TreeSpecies',
 'form2.TriageCategory',
 'form2.TreeHeight',
 'form2.CrownSpread',
 'form2.NoOfTr

In [ ]:
df_dbtreeList= pd.json_normalize(datameta, record_path = ['treeList'])
df_dbtreeList

,ID,TreeID,DEPTTreeID,SpeciesCode,SpeciesTypeCode,MasterZoneID,SubZoneID,TriageColourCode,TriageColourTypeCode,X,Y,isNewTree
0,a2737c38-fb6f-46cf-867e-0000b525f6d2,202003200081225,RTR0002772,514,TreeSpecies,6383,18651,None,None,837747.133,813843.561,False
1,61f7fd85-225a-4227-a6bc-00026d410def,202003200040625,RTR0020104,236,TreeSpecies,9386,21653,None,None,838568.832,810496.441,False
2,fd3b4478-e880-4fa0-82e9-000329bb800b,202003200087798,RTR0013299,169,TreeSpecies,6693,18961,None,None,841592.097,813138.167,False
3,34d79974-bcce-40ce-99d2-0004d45bd9f4,202003200088745,RTR0094282,473,TreeSpecies,9774,22041,None,None,835241.977,811277.595,False
4,444425f0-1041-4c97-bfcf-0005309a7211,202003200053564,RTR0097323,138,TreeSpecies,7197,19465,None,None,834158.013,815595.844,False
...,...,...,...,...,...,...,...,...,...,...,...,...
55862,7ced2e0c-1c84-4f45-a38b-fffe5520ebd5,202003200004177,RTR0019841,335,TreeSpecies,9460,21727,None,None,841102.751,810661.118,False
55863,d109f5db-1eff-48a4-a6b0-fffe90d9feb1,202003200010662,RTR0020449,111,TreeSpecies,9427,21694,None,None,841076.869,810863.337,False
55864,3690638f-6e80-49a3-8094-fffefa36054f,202003200191936,RTR0035265,333,TreeSpecies,7600,19867,None,None,831524.012,813922.678,False
55865,eea985d6-b11b-4b91-b5f5-ffff4e8937a7,202103200000015,ETR0630012,229,TreeSpecies,29672,49723,None,None,833739.073,815473.083,False


In [ ]:
zipfile

<module 'zipfile' from 'C:\\Users\\ckho\\Desktop\\Soft\\Jupyter_Portable\\JuypterPortable\\apps\\lib\\zipfile.py'>

In [ ]:
dferrorlist = pd.DataFrame(errlist)
dferrorlist.to_excel(xlErrPath)

In [ ]:
errlist

[]

## cont

In [ ]:
dall.keys()

dict_keys(['attachment', 'tblLocationTypeList', 'tblForm2TargetAssessmentList', 'tblForm2BranchConditionsList', 'tblForm2MitigationMeasuresList', 'tblForm2PruningHistoryList', 'tblForm2RootConditionsList', 'tblForm2TrunkConditionList', 'tblForm2TreeOnTreeRegisterList', 'tblForm2InspectionLimitationsList', 'tblForm2DiebackTwigsTypeList', 'tblForm2LeanList', 'tblForm2SiteChangesList', 'tblForm2SoilConditionsList', 'tblForm2TopographyList', 'tblForm2AuditCommentList', 'auditTabs'])

In [ ]:
#dall['tblForm2MitigationMeasuresList']

## 除中文字

In [ ]:
df_f2basicAll.columns.to_list()

['MasterZoneNo',
 'DateOfInspection',
 'form2.FormSubversion',
 'form2.ID',
 'form2.TreeGUID',
 'form2.ContractGUID',
 'form2.FileRef',
 'form2.DepartmentGUID',
 'form2.InspectionOfficerGUID',
 'form2.DateOfInspection',
 'form2.LastInspectionTime',
 'form2.InspectionFrequency',
 'form2.InspectionTimeSpent',
 'form2.LastExportDatetime',
 'form2.LastImportDatetime',
 'form2.LastUploadDatetime',
 'form2.TreeSpecies',
 'form2.TriageCategory',
 'form2.TreeHeight',
 'form2.CrownSpread',
 'form2.NoOfTrunk',
 'form2.DBHOfTreeTrunk1',
 'form2.DBHOfTreeTrunk2',
 'form2.DBHOfTreeTrunk3',
 'form2.DBHOfTreeTrunk4',
 'form2.DBHOfTreeTrunk5',
 'form2.AggregatedDBH',
 'form2.TreeRegisterNo',
 'form2.OVTTreeRegisterNo',
 'form2.FormRef',
 'form2.MasterZoneGUID',
 'form2.SubZoneGUID',
 'form2.EnglishLocation',
 'form2.ChineseLocation',
 'form2.Category',
 'form2.District',
 'form2.CoordinateX',
 'form2.CoordinateY',
 'form2.NearestLampPoleNumber',
 'form2.SoilCrackOrCrackBehindLean',
 'form2.SoilCrackOr

In [ ]:
df_f2basicAll['form2.DepartmentGUID'] #=> form2.DepartmentGUID relate to treeList [ID]

0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
0    b9604f7e-c99e-415b-8259-ffa2655783e4
Name: form2.DepartmentGUID, dtype: object

In [ ]:
df_f2basicSummaryAll = df_f2basicAll[['form2.DEPTTreeID',
 'MasterZoneNo',
 'DateOfInspection',
 'form2.FormSubversion',
 'form2.FileRef',
 'form2.DateOfInspection',
 'form2.LastInspectionTime',
 'form2.InspectionFrequency',
 'form2.TreeSpecies',
 'form2.TriageCategory',
 'form2.TreeHeight',
 'form2.CrownSpread',
 'form2.AggregatedDBH',
 'form2.TreeVigor',
 'form2.AngleFromVertical',
 'form2.OverallTreeRiskRating',
 'form2.OverallResidualRisk',
 'form2.CoordinateX',
 'form2.CoordinateY',
 'form2.SoilCrackOrCrackBehindLean',
 'form2.RestrictionWithinDripline',
 'form2.NextInspectionDate',
 'form2.InspectionOfficerNameEnglish'
]]

df_mitigationAll = dall['tblForm2MitigationMeasuresList'].iloc[:,2:]       #自訂義勾其他columns
df_TreeRegAll = dall['tblForm2TreeOnTreeRegisterList'].iloc[:,2:]

df_f2SummaryResult = df_f2basicSummaryAll.merge(df_mitigationAll, left_on='form2.DEPTTreeID', right_on='TreeID')
df_f2SummaryResult = df_f2SummaryResult.merge(df_TreeRegAll, left_on='TreeID', right_on='TreeID')

# remove ENTER tab
df_f2SummaryResult.MitigationMeasures = df_f2SummaryResult.MitigationMeasures.str.replace('\r\n', ';').str[:-1]
df_f2SummaryResult = df_f2SummaryResult.iloc[:,1:]
df_f2SummaryResult

,MasterZoneNo,DateOfInspection,form2.FormSubversion,form2.FileRef,form2.DateOfInspection,form2.LastInspectionTime,form2.InspectionFrequency,form2.TreeSpecies,form2.TriageCategory,form2.TreeHeight,...,ResidualRisk,TreeID,TMCPTreeID_x,OverallComment_x,fldPath_x,TreeRegisterType,Remark,TMCPTreeID_y,OverallComment_y,fldPath_y
0,11SW-A/R577_0,2024-10-28T15:10:27,1.2,ETR0610143_11SW-AR577_20241028,2024-10-28T15:10:27,2024-04-29T00:00:00,6 months,Ficus microcarpa,Red,4.0,...,Moderate,ETR0610143,202403200001163,The tree is situated on a retaining wall besid...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...,cbxForm2_TreeInfoST,None,202403200001163,The tree is situated on a retaining wall besid...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...
1,11SW-A/R577_0,2024-10-28T15:10:27,1.2,ETR0610143_11SW-AR577_20241028,2024-10-28T15:10:27,2024-04-29T00:00:00,6 months,Ficus microcarpa,Red,4.0,...,Moderate,ETR0610143,202403200001163,The tree is situated on a retaining wall besid...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...,cbxForm2_TreeInfoCS,None,202403200001163,The tree is situated on a retaining wall besid...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...
2,11SW-A/R577_0,2024-10-28T15:10:27,1.2,ETR0610143_11SW-AR577_20241028,2024-10-28T15:10:27,2024-04-29T00:00:00,6 months,Ficus microcarpa,Red,4.0,...,Moderate,ETR0610143,202403200001163,The tree is situated on a retaining wall besid...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...,cbxForm2_TreeInfoST,None,202403200001163,The tree is situated on a retaining wall besid...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...
3,11SW-A/R577_0,2024-10-28T15:10:27,1.2,ETR0610143_11SW-AR577_20241028,2024-10-28T15:10:27,2024-04-29T00:00:00,6 months,Ficus microcarpa,Red,4.0,...,Moderate,ETR0610143,202403200001163,The tree is situated on a retaining wall besid...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...,cbxForm2_TreeInfoCS,None,202403200001163,The tree is situated on a retaining wall besid...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...
4,11SW-A/R577_0,2024-10-28T15:00:17,1.2,ETR0610144_11SW-AR577_20241028,2024-10-28T15:00:17,2024-04-29T00:00:00,6 months,Ficus microcarpa,Red,5.0,...,Moderate,ETR0610144,202403200001161,\nThe tree is situated on a retaining wall bes...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...,cbxForm2_TreeInfoCS,None,202403200001161,\nThe tree is situated on a retaining wall bes...,L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,11SW-A/R602_0,2024-10-28T14:45:16,1.2,RTR0033994_11SW-AR602_20241028,2024-10-28T14:45:16,2023-10-20T00:00:00,6 months,Ficus microcarpa,Red,10.0,...,Low,RTR0033994,202003200188966,"The tree is situated on a retaining wall, with...",L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...,cbxForm2_TreeInfoMT,None,202003200188966,"The tree is situated on a retaining wall, with...",L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...
57,11SW-A/R602_0,2024-10-28T14:45:16,1.2,RTR0033994_11SW-AR602_20241028,2024-10-28T14:45:16,2023-10-20T00:00:00,6 months,Ficus microcarpa,Red,10.0,...,Low,RTR0033994,202003200188966,"The tree is situated on a retaining wall, with...",L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...,cbxForm2_TreeInfoST,None,202003200188966,"The tree is situated on a retaining wall, with...",L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...
58,11SW-A/R602_0,2024-10-28T14:11:27,1.2,RTR0034003_11SW-AR602_20241028,2024-10-28T14:11:27,2024-04-29T00:00:00,6 months,Ficus microcarpa,Red,13.0,...,Moderate,RTR0034003,202003200188972,"The tree was positioned on a retaining wall, w...",L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...,cbxForm2_TreeInfoMT,None,202003200188972,"The tree was positioned on a retaining wall, w...",L:\LU\Staff\Ricky\TS\01-OVT+SWT\15-Batch 15 (2...
59,11SW-A/R602_0,2024-10-28T14:11:27,1.2,RTR0034003_11SW-AR602_20241028,2024-10-28T14:11:27,2024-04-29T00:00:00,6 months,Ficus microcarpa,Red,13.0,...,Moderate,RTR0034003,2

In [ ]:
#除中文字
# https://stackoverflow.com/questions/2718196/find-all-chinese-text-in-a-string-using-python-and-regex

df_f2SummaryResult['MitigationMeasures'] = df_f2SummaryResult['MitigationMeasures'].astype(str)

import re
sublist = []
for index, row in df_f2SummaryResult.iterrows():

    cell = []
    for n in re.findall(r'[^\u4e00-\u9fff]+', row['MitigationMeasures']):

        cell.append(n)
        #print(cell)

    #print(row['c1'], row['c2'])
    sublist.append(cell)

print(sublist)

[['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring;Crown reduction  ', ';Crown thinning  '], ['Others  ', 'Monitoring;Crown reduction  ', ';Crown thinning  '], ['Others  ', 'Monitoring;Crown thinning  ', ';Crown reduction  '], ['Others  ', 'Monitoring;Crown thinning  ', ';Crown reduction  '], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Crown thinning  ', ';Crown reduction  '], ['Crown thinning  ', ';Crown reduction  '], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Crown cleaning  '], ['Crown cleaning  '], ['Crown cleaning  '], ['Others  ', 'monitoring'], ['Others  ', 'monitoring'], ['Others  ', 'monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Monitoring'], ['Others  ', 'Mon

In [ ]:
df_f2SummaryResult['MitigationMeasures_Eng'] = sublist
df_f2SummaryResult['MitigationMeasures_Eng'] = df_f2SummaryResult['MitigationMeasures_Eng'].apply(lambda x: ', '.join(map(str, x)))
df_f2SummaryResult['MitigationMeasures_Eng'] = df_f2SummaryResult['MitigationMeasures_Eng'].str.replace("Others  ,", "Others:")
df_f2SummaryResult

df_f2SummaryResult.to_excel(xlExportPath)

## staff id

In [ ]:
df_summarymeta = pd.json_normalize(datameta)
df_summarymeta

,masterValueList,contractList,departmentList,inspectionOfficerList,masterZoneList,subZoneList,treeList,csvVersion.GUID,csvVersion.WINCLIENT_DATE,csvVersion.EXPORT_DATETIME
0,[{'ID': '9ec10ef5-fef8-40db-bffc-000567b0e871'...,[{'ID': 'ba803d1d-f36b-4b60-ac63-03baa1967e01'...,[{'ID': '2edf1c55-e512-4600-879a-115d4ce1ef76'...,[{'ID': '8ae7020b-27b4-4d75-a3aa-05b713c442e5'...,[{'ID': '47fd98d1-87d4-40f9-b7dc-00068aa47b3a'...,[{'ID': '998b009a-467d-4f36-bfe5-000a04cce7a4'...,[{'ID': 'a2737c38-fb6f-46cf-867e-0000b525f6d2'...,42c909ce-e472-43af-a32c-93d8c3b429bc,2025-01-14T10:53:00,2025-01-14T11:34:00


In [ ]:
df_staffGUID = pd.json_normalize(datameta, record_path = ['inspectionOfficerList'])

df_staffGUID['InspectionOfficerID'] = df_staffGUID['InspectionOfficerID'].astype(str)
df_staffGUID.head()

,ID,InspectionOfficerID
0,8ae7020b-27b4-4d75-a3aa-05b713c442e5,4119
1,33977358-ebcd-4c81-81b8-09888a35b2ed,4185
2,e8d17e08-0699-46d7-8422-2012b2864edf,3535
3,c417ddea-0d6b-499f-ace6-230224e06458,4049
4,a699d798-ae6e-4548-8c4d-2967638c7c5c,1352


In [ ]:
dCsv['TBL_INSPECTOR.csv']['INSP_OF_OFFICER_ID'] = dCsv['TBL_INSPECTOR.csv']['INSP_OF_OFFICER_ID'].astype(str)
dCsv['TBL_INSPECTOR.csv'].head()

,ID,CSV_CODE,INSP_OF_OFFICER_ID,INSP_OF_NAME_ENG,INSP_OF_NAME_CHI,INSP_OF_POST,INSP_OF_UNIQUE_ID,LU_DEPT_CODE,INSP_OF_EMAIL_ADDRESS,INSP_OF_CONTRACT_ID,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,4043,NaN,4043,CHAN CHUEN KIU,陳泉橋,Assistant Arborist,AA14,32,chanchuenkiu@sy12hy19.com.hk,1794,12/06/2024 15:58,01/14/2025 11:34,Y,1
1,839,NaN,839,"CHAN HO MAN, TIMOTHY",陳灝文,Assistant Arborist,AA07,32,hmchan@sy12hy19.com.hk,1794,12/06/2024 15:57,01/14/2025 11:34,Y,2
2,4044,NaN,4044,CHAN KING HEI,陳景曦,Arborist,CHANKINGHEI,32,davy.chan@baguio.com.hk,1794,12/06/2024 15:58,01/14/2025 11:34,Y,3
3,4119,NaN,4119,CHAN Shuk Han,NaN,Arborist,CHANSHUKHAN,32,NaN,1794,12/06/2024 15:58,01/14/2025 11:34,Y,4
4,840,NaN,840,"CHAN SIU LUNG, TANGO",陳小龍,Assistant Arborist,AA08,32,slchan@sy12hy19.com.hk,1794,12/06/2024 15:57,01/14/2025 11:34,Y,5


In [ ]:
df_staff = pd.merge(df_staffGUID, dCsv['TBL_INSPECTOR.csv'], left_on='InspectionOfficerID', right_on='INSP_OF_OFFICER_ID')
df_staff

,ID_x,InspectionOfficerID,ID_y,CSV_CODE,INSP_OF_OFFICER_ID,INSP_OF_NAME_ENG,INSP_OF_NAME_CHI,INSP_OF_POST,INSP_OF_UNIQUE_ID,LU_DEPT_CODE,INSP_OF_EMAIL_ADDRESS,INSP_OF_CONTRACT_ID,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,8ae7020b-27b4-4d75-a3aa-05b713c442e5,4119,4119,NaN,4119,CHAN Shuk Han,NaN,Arborist,CHANSHUKHAN,32,NaN,1794,12/06/2024 15:58,01/14/2025 11:34,Y,4
1,33977358-ebcd-4c81-81b8-09888a35b2ed,4185,4185,NaN,4185,Chu Ching Ki,NaN,Arborist,CHUCHINGKI,32,NaN,1794,01/06/2025 12:03,01/14/2025 11:34,Y,9
2,c417ddea-0d6b-499f-ace6-230224e06458,4049,4049,NaN,4049,FUNG Ho Tsang,馮浩錚,Arborist,FUNGHOTSANG,32,hftsoi@winghoyuen.com,1794,12/06/2024 15:59,01/14/2025 11:34,Y,11
3,fce16a15-b137-40e0-986d-2b240d4c01b8,4048,4048,NaN,4048,YUEN PIK YEE,袁碧兒,Arborist,YUENPIKYEE,32,lynn.yuen@baguio.com.hk,1794,12/06/2024 15:59,01/14/2025 11:34,Y,32
4,e116448b-9d88-45ee-8401-4d549d955ba0,4046,4046,NaN,4046,LUK KA CHUN,陸加俊,Arborist,LUKKACHUN,32,fung.cheng@baguio.com.hk,1794,12/06/2024 15:59,01/14/2025 11:34,Y,21
5,59303722-6cf1-4fa4-83e1-53387bbbe452,1722,1722,NaN,1722,"WU SIN CHING, CATHERINE",胡倩菁,Arborist,AR08,32,scwu@sy12hy19.com.hk,1794,12/06/2024 15:58,01/14/2025 11:34,Y,27
6,694061e7-645a-4f8a-9b53-5a6b118fe400,4121,4121,NaN,4121,WAN Wut Hang,NaN,Arborist,WANWUTHANG,32,NaN,1794,12/06/2024 15:59,01/14/2025 11:34,Y,24
7,e5366768-1631-47f1-977c-62792d619475,4149,4149,NaN,4149,Cheung Wang,NaN,Arborist,CHEUNGWANG,32,NaN,1794,12/17/2024 11:53,01/14/2025 11:34,Y,8
8,df75d7aa-8d7d-4855-8746-64ba4a1b00b8,532,532,NaN,532,"LAW CHI KA, CHRIS",羅志嘉,Tree Specialist,TS02,32,cklaw@sy12hy19.com.hk,1794,12/06/2024 15:59,01/14/2025 11:34,Y,18
9,172bc8bb-43dc-4a21-8075-68a02e6c1532,839,839,NaN,839,"CHAN HO MAN, TIMOTHY",陳灝文,Assistant Arborist,AA07,32,hmchan@sy12hy19.com.hk,1794,12/06/2024 15:57,01/14/2025 11:34,Y,2


In [ ]:
df_f2basicAll['form2.InspectionOfficerFullName']

0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
0    TS04-HO CHAK KONG(Tree Specialist),ckho@sy12hy...
Name: form2.InspectionOfficerFullName, dtype: object

### inspector GUID

In [ ]:
df_f2basicAll[['form2.InspectionOfficerGUID']].iloc[0,0]

'2902076d-de6a-48ee-991d-e8aacdaf4e02'

In [ ]:
###=======================================================================###

# Meta

# Read CSVtoMasterValue json

In [ ]:
jsondata = r'C:\TMCP\CSVToMasterValue.json'

pathlist = str(Path(jsondata).parent)
pathlist

with open(jsondata) as f:
    json_data = f.read()
    data = json.loads(json_data)
    #print(data)
    df_csv = pd.json_normalize(data)

df_csv['link'] = pathlist + '\\' +df_csv['FileName']
df_csv.head()

,FileName,MasterType,IsSystem,link
0,TBL_ADVANCED_ASSESSMENT.csv,AdvancedAssessment,false,C:\TMCP\TBL_ADVANCED_ASSESSMENT.csv
1,TBL_ASPECT.csv,Aspect,false,C:\TMCP\TBL_ASPECT.csv
2,TBL_AUDITOR.csv,,false,C:\TMCP\TBL_AUDITOR.csv
3,TBL_CONTRACT.csv,,False,C:\TMCP\TBL_CONTRACT.csv
4,TBL_CROWN_DENSITY_PERCENT.csv,CrownDensityD,false,C:\TMCP\TBL_CROWN_DENSITY_PERCENT.csv


# Read Version and Mapping

In [ ]:
jsondata = r'C:\Users\ckho\Desktop\tmcp\VersionAndMapping.json'

with open(jsondata) as f:
    json_data = f.read()
    data = json.loads(json_data)
    print(data)
    df_summary = pd.json_normalize(data)

df_dbmasterValue = pd.json_normalize(data, record_path = ['masterValueList'])
df_dbmasterValue.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ckho\\Desktop\\tmcp\\VersionAndMapping.json'

In [ ]:
#cb clickbox
df_decode1 = df_dbmasterValue[df_dbmasterValue.Code.str.startswith('cb')]
df_decode1 = df_decode1.rename(columns={"Code": "Result", "MasterTypeCode": "Code"})
df_decode1

In [ ]:
#form2.SoilCrackOrCrackBehindLean
#form2.Status

otherlist = ['SCOrCBLean','Status']

df_decode2 = df_dbmasterValue[(df_dbmasterValue.MasterTypeCode.isin(otherlist))]
df_decode2 = df_decode2.rename(columns={"Code": "Result", "MasterTypeCode": "Code"})
df_decode2.head()

In [ ]:
df_dbmasterValue.head()

In [ ]:
df_summary.head()

# Merge TwoDataset

In [ ]:
dflink = pd.merge(df_dbmasterValue,df_csv, left_on='MasterTypeCode', right_on='MasterType')
#dflink['Code'].astype(str)
dflink.head()

# Decoding Mapping and Version 'Result Column'

In [ ]:
tmcppath = r'C:\TMCP\CSVUnZIPTemp'

#create dataframe blank in dic
d=None
d={}

path = pathlib.Path(tmcppath)
files = path.glob("**/TBL*.csv")

#codetablelist
for i in files:
    d[i.stem + i.suffix] = pd.read_csv(i, sep='\t')
d.keys()

## Decoding

In [ ]:
resultlist = []
for index, row in dflink.iterrows():
    #print(row['Code'])
    selectData = [int(row['Code'])]
    #print(d[row["FileName"]])
    dfa = d[row["FileName"]]
    dfacode = d[row["FileName"]]['CSV_CODE']
    result = dfa[(dfacode.isin(selectData))].iloc[0,3]
    resultlist.append(result)
    #df2[(df2['ID'].isin(df1['ID']))]

dflink['Result'] = resultlist

dflink = pd.concat([dflink, df_decode1, df_decode2], ignore_index=True)
dflink

# Read F2 json and Decoding

In [ ]:
jsondata = r'C:\Users\ckho\Desktop\tmcp\RTR0015158_11SW-B-R732_1_20240517.json'

with open(jsondata, encoding="utf-8") as f:
    json_data = f.read()
    data = json.loads(json_data)
    #print(data)
    df_summary = pd.json_normalize(data)

In [ ]:
# empty df
df_f2basicAll = []

## Section Basic infomation

In [ ]:
f2basiclist =  df_summary.columns.to_list()[18:]
#f2basiclist

In [ ]:
df_f2basic = df_summary[f2basiclist].replace(dflink.ID.tolist(),  dflink.Result.tolist())
df_f2basic

### concat, change df_f2basicAll to pandas

In [ ]:
TreeID = df_f2basic['form2.DEPTTreeID'].iloc[0]
TreeID

In [ ]:
count = 0
if count == 0:
    df_f2basicAll = pd.DataFrame(columns = df_f2basic.columns.tolist())
df_f2basicAll = pd.concat([df_f2basicAll, df_f2basic])
df_f2basicAll

In [ ]:
#df_f2basic.columns.to_list()

In [ ]:
#df_f2basic.to_excel(r'C:\Users\ckho\Desktop\tmcp\basic.xlsx')

## Section csv

In [ ]:
num = 0

#create dataframe blank in dic
d = None
d = {}
dall = None
dall = {}

f2sectionlist = df_summary.columns.to_list()[0:17]

#codetablelist
for i in f2sectionlist:
    d[i] = pd.json_normalize(data, record_path = [i])
    d[i]['TreeID']=TreeID

    #print(num)

    ###Decoding
    d[i] = d[i].replace(dflink.ID.tolist(),  dflink.Result.tolist())

    #Append gp
    if num == 0:
        dall[i] = pd.DataFrame(columns = d[i].columns.tolist())
    dall[i] = pd.concat([dall[i],d[i]])

num = num +1
d.keys()
dall.keys()